# Notebook 1: NER Fundamentals

**Named Entity Recognition (NER)** is the task of identifying spans of text that refer to real-world entities and classifying them into predefined categories.

## What you'll learn:
1. What NER is and why it matters
2. Common entity types (PERSON, ORG, GPE, DATE, etc.)
3. How to use spaCy for out-of-the-box NER
4. Visualizing entities with displaCy
5. Understanding BIO/IOB tagging schemes
6. Evaluating NER performance

## 1. What is Named Entity Recognition?

NER is a subtask of **Information Extraction** that locates and classifies named entities in unstructured text.

**Example:**
```
Input:  "Barack Obama visited Google headquarters in Mountain View."
Output: [Barack Obama -> PERSON] [Google -> ORG] [Mountain View -> GPE]
```

### Why does NER matter?
- **Search engines** - Understanding queries
- **Question answering** - Extracting answers from documents
- **Knowledge graphs** - Building structured data from text
- **Healthcare** - Extracting drug names, diseases, symptoms from clinical notes
- **Finance** - Identifying companies, monetary values, dates in reports

## 2. Setup - Install and Load spaCy

In [ ]:
# Install spaCy and download the English model (run once)
# !pip install spacy
# !python -m spacy download en_core_web_sm

import spacy
from spacy import displacy

# Load the small English model
nlp = spacy.load("en_core_web_sm")
print(f"spaCy version: {spacy.__version__}")
print(f"Model: en_core_web_sm")
print(f"Pipeline components: {nlp.pipe_names}")

## 3. Your First NER Example

Let's process a sentence and extract all named entities.

In [ ]:
# Process a sentence through the NLP pipeline
text = "Apple was founded by Steve Jobs in Cupertino, California in 1976."
doc = nlp(text)

# Extract entities
print("Entities found:")
print("-" * 50)
for ent in doc.ents:
    print(f"  {ent.text:20s}  |  Label: {ent.label_:10s}  |  Description: {spacy.explain(ent.label_)}")

print(f"\nTotal entities found: {len(doc.ents)}")

## 4. Common Entity Types in spaCy

spaCy's `en_core_web_sm` model recognizes 18 entity types. Here are the most common ones:

| Label | Description | Example |
|-------|-------------|--------|
| PERSON | People, including fictional | *Steve Jobs* |
| ORG | Companies, agencies, institutions | *Apple Inc.* |
| GPE | Countries, cities, states | *California* |
| DATE | Absolute or relative dates | *June 2024* |
| MONEY | Monetary values | *$1 million* |
| LOC | Non-GPE locations | *Mount Everest* |
| PRODUCT | Objects, vehicles, foods | *iPhone* |
| EVENT | Named events | *World War II* |
| CARDINAL | Numerals | *three* |
| ORDINAL | "first", "second", etc. | *first* |

In [ ]:
# List ALL entity types the model knows about
print("All entity labels in this model:")
print("-" * 50)
for label in nlp.get_pipe("ner").labels:
    print(f"  {label:12s} -> {spacy.explain(label)}")

## 5. Visualizing Entities with displaCy

spaCy has a built-in visualizer called **displaCy** that renders entities beautifully in Jupyter notebooks.

In [ ]:
# Visualize entities inline in the notebook
text = """Elon Musk, the CEO of Tesla and SpaceX, met with President Biden
at the White House on January 15, 2024. They discussed a $2 billion
investment in renewable energy infrastructure across the United States."""

doc = nlp(text)
displacy.render(doc, style="ent", jupyter=True)

In [ ]:
# You can filter to show only specific entity types
displacy.render(doc, style="ent", jupyter=True, options={"ents": ["PERSON", "ORG"]})

## 6. Understanding BIO/IOB Tagging

NER models don't just label whole entities - they label each **token** using the **BIO scheme**:

- **B-TAG** = **B**eginning of an entity
- **I-TAG** = **I**nside (continuation of) an entity
- **O** = **O**utside any entity

**Example:**
```
Token:   Steve   Jobs   founded   Apple   in   Cupertino
BIO:     B-PER   I-PER  O         B-ORG   O    B-GPE
```

This scheme allows the model to handle:
- Multi-word entities ("Steve Jobs" = B-PER + I-PER)
- Adjacent entities of the same type

In [ ]:
# See BIO tags for each token in spaCy
text = "Steve Jobs founded Apple in Cupertino."
doc = nlp(text)

print(f"{'Token':15s} {'IOB Tag':10s} {'Entity Type':12s} {'IOB (numeric)'}")
print("-" * 55)
for token in doc:
    print(f"{token.text:15s} {token.ent_iob_:10s} {token.ent_type_:12s} {token.ent_iob}")

print("\nIOB codes: 2=Outside, 3=Begin, 1=Inside")

## 7. NER on Multiple Sentences (Batch Processing)

spaCy's `nlp.pipe()` is efficient for processing many texts at once.

In [ ]:
import pandas as pd

texts = [
    "Microsoft acquired LinkedIn for $26.2 billion in December 2016.",
    "Sundar Pichai became CEO of Alphabet in December 2019.",
    "The European Union fined Google 4.34 billion euros in 2018.",
    "Jeff Bezos stepped down as Amazon CEO on July 5, 2021.",
]

# Batch process
results = []
for doc in nlp.pipe(texts):
    for ent in doc.ents:
        results.append({
            "Text": ent.text,
            "Label": ent.label_,
            "Start": ent.start_char,
            "End": ent.end_char,
            "Source": doc.text[:50] + "..."
        })

df = pd.DataFrame(results)
print(f"Total entities extracted: {len(df)}")
df

## 8. Entity Frequency Analysis

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

# Count entity types
label_counts = Counter(df["Label"])

plt.figure(figsize=(8, 4))
plt.bar(label_counts.keys(), label_counts.values(), color="steelblue")
plt.title("Entity Type Distribution")
plt.xlabel("Entity Type")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

## 9. Evaluating NER - Precision, Recall, F1

NER evaluation compares **predicted** entities against **gold standard** (true) entities.

- **Precision** = Of all entities the model predicted, how many were correct?
- **Recall** = Of all true entities, how many did the model find?
- **F1 Score** = Harmonic mean of Precision and Recall

An entity is considered **correct** only if BOTH the span AND the label match exactly.

In [ ]:
# Manual evaluation example
gold_entities = {("Steve Jobs", "PERSON"), ("Apple", "ORG"), ("Cupertino", "GPE")}
pred_entities = {("Steve Jobs", "PERSON"), ("Apple", "ORG"), ("California", "GPE")}

tp = gold_entities & pred_entities
fp = pred_entities - gold_entities
fn = gold_entities - pred_entities

precision = len(tp) / (len(tp) + len(fp)) if (len(tp) + len(fp)) > 0 else 0
recall = len(tp) / (len(tp) + len(fn)) if (len(tp) + len(fn)) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"True Positives:  {tp}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"\nPrecision: {precision:.2f}")
print(f"Recall:    {recall:.2f}")
print(f"F1 Score:  {f1:.2f}")

## 10. Exercises

Try these on your own:

1. **Extract entities from a news article**: Copy-paste a paragraph from any news site and run NER on it.
2. **Entity type breakdown**: Process 10+ sentences and create a bar chart of entity type frequencies.
3. **Compare models**: Load `en_core_web_md` (medium) and compare its NER results to `en_core_web_sm`.

---

**Next**: [Notebook 2 - Custom NER Training with spaCy](02_custom_ner_spacy.ipynb)